# Executed notebook: E09_source_integrity

Exported from Databricks job run `994447175034582` (task `E09_source_integrity`, task run `684494378934229`).

Result: **SUCCESS** · start 2026-10-07T23:14:02.443000+00:00 · end 2026-10-07T23:14:20.500000+00:00

Run URL: https://fevm-serverless-stable-am1uc2.cloud.databricks.com/?o=7474651880045550#job/755461157363253/run/994447175034582


# E09 · Source integrity: the code that runs is the code in the repo

`evidence/source_snapshot/` holds a byte-identical copy of the pipeline, shared feature/physics code, simulator and
pipeline config, with `SHA256SUMS` (written by `tools/snapshot_source.py`).

**What this notebook proves.**
1. The running SDP pipeline's libraries point to the bundle-deployed files.
2. The SHA-256 of every **deployed** file equals the committed checksum (and the committed snapshot copy).
3. The full deployed pipeline source, printed below as text.

In [ ]:
%pip install -q "databricks-sdk>=0.81" pg8000
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
from _helpers import *  # noqa: F401,F403
import hashlib, os
ROOT = os.path.abspath("../../..")  # bundle files root in the workspace
print("deployed bundle root:", ROOT)

deployed bundle root: /Workspace/Users/sascha.vetter@databricks.com/.bundle/mfg-predictive-maintenance/demo/files


## 1 · Pipeline libraries as configured on the running pipeline

In [ ]:
p = w.pipelines.get(PIPELINE_ID)
libs = [l.file.path for l in p.spec.libraries]
print("pipeline:", p.name, "| state:", p.state.value, "| continuous:", p.spec.continuous)
for l in libs:
    print("  library:", l, "| under deployed root:", l.startswith(ROOT.replace("/Workspace", "")) or l.startswith(ROOT))

pipeline: pdm_live_pipeline | state: RUNNING | continuous: True
  library: /Workspace/Users/sascha.vetter@databricks.com/.bundle/mfg-predictive-maintenance/demo/files/src/pipeline/01_sensor_readings_clean.py | under deployed root: True
  library: /Workspace/Users/sascha.vetter@databricks.com/.bundle/mfg-predictive-maintenance/demo/files/src/pipeline/02_station_risk_scores.py | under deployed root: True
  library: /Workspace/Users/sascha.vetter@databricks.com/.bundle/mfg-predictive-maintenance/demo/files/src/pipeline/03_station_health_current.py | under deployed root: True
  library: /Workspace/Users/sascha.vetter@databricks.com/.bundle/mfg-predictive-maintenance/demo/files/src/pipeline/04_maintenance_events.py | under deployed root: True


## 2 · SHA-256 of deployed files vs committed checksums vs committed snapshot

In [ ]:
sums = {}
for line in open(os.path.join(ROOT, "evidence/source_snapshot/SHA256SUMS")).read().splitlines():
    h, f = line.split("  ", 1)
    sums[f] = h
sha = lambda path: hashlib.sha256(open(path, "rb").read()).hexdigest()
rows = []
for f, expected in sorted(sums.items()):
    deployed = sha(os.path.join(ROOT, f))
    snapshot = sha(os.path.join(ROOT, "evidence/source_snapshot", f))
    rows.append((f, deployed[:16], expected[:16], snapshot[:16], deployed == expected == snapshot))
display(spark.createDataFrame(rows, "file string, deployed_sha256 string, committed_sha256 string, snapshot_sha256 string, identical boolean"))
print("ALL IDENTICAL" if all(r[-1] for r in rows) else "MISMATCH FOUND")

| file | deployed_sha256 | committed_sha256 | snapshot_sha256 | identical |
|---|---|---|---|---|
| resources/pipeline.yml | be17f976a535d73e | be17f976a535d73e | be17f976a535d73e | True |
| src/pdm/__init__.py | e3b0c44298fc1c14 | e3b0c44298fc1c14 | e3b0c44298fc1c14 | True |
| src/pdm/config.py | fdbb712446294c2b | fdbb712446294c2b | fdbb712446294c2b | True |
| src/pdm/features.py | e599eb4f32a94daf | e599eb4f32a94daf | e599eb4f32a94daf | True |
| src/pdm/physics.py | c69618599cb93d07 | c69618599cb93d07 | c69618599cb93d07 | True |
| src/pipeline/01_sensor_readings_clean.py | 217fb117f1a001b7 | 217fb117f1a001b7 | 217fb117f1a001b7 | True |
| src/pipeline/02_station_risk_scores.py | 7783a6ba6b0ca9fb | 7783a6ba6b0ca9fb | 7783a6ba6b0ca9fb | True |
| src/pipeline/03_station_health_current.py | 3b0149c23ec384b9 | 3b0149c23ec384b9 | 3b0149c23ec384b9 | True |
| src/pipeline/04_maintenance_events.py | 35714378c3c0483b | 35714378c3c0483b | 35714378c3c0483b | True |
| src/pipeline/_common.py | 0f

ALL IDENTICAL


## 3 · Full deployed pipeline source

In [ ]:
for f in sorted(k for k in sums if k.startswith("src/pipeline/") or k == "src/pdm/features.py" or k == "resources/pipeline.yml"):
    print(f"{'=' * 100}\n# {f}  sha256={sums[f]}\n{'=' * 100}")
    print(open(os.path.join(ROOT, f)).read())

# resources/pipeline.yml  sha256=be17f976a535d73e51cd5b75fb78af81139cb1d9e100d8544fa34cac54676689
resources:
  pipelines:
    pdm_live:
      name: pdm_live_pipeline
      catalog: ${var.catalog}
      schema: pdm_core
      serverless: true
      continuous: true
      photon: true
      channel: CURRENT
      root_path: ../src/pipeline
      libraries:
        - file:
            path: ../src/pipeline/01_sensor_readings_clean.py
        - file:
            path: ../src/pipeline/02_station_risk_scores.py
        - file:
            path: ../src/pipeline/03_station_health_current.py
        - file:
            path: ../src/pipeline/04_maintenance_events.py
      configuration:
        pdm.catalog: ${var.catalog}
        pdm.source_path: ${workspace.file_path}/src
      environment:
        dependencies:
          - scikit-learn==1.7.2
          - mlflow>=3.1

# src/pdm/features.py  sha256=e599eb4f32a94daf4c7f5448d30c649123370b4ed0ff93ae4c4a60ff21908122
"""Single feature definition shar